In [19]:

from pathlib import Path
import csv
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from models.multi_attention_unet import MutliRepUnet
from src.lidar_dataset import LidarDataset
from src.metrics import segmentation_metrics, calculate_metrics


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

REPRESENTATION = 'combined'
BATCH_SIZE = 2
NUM_WORKERS = 0
BEST_THRESHOLDS = [0.35, 0.375, 0.3]
REPRESENTATION_NAMES = ['SVF', 'Openness', 'Slope']
CLASS_NAMES = ['Building', 'Platform', 'Aguada']

EXPERIMENT_NAME = 'multi_rep_unet'

MANIFEST_PATH = (PROJECT_ROOT / 'data/processed/manifest_split.csv')

CHECKPOINT_PATH = (PROJECT_ROOT /'checkpoints' /f'{EXPERIMENT_NAME}.pt')

OUTPUT_DIR = (PROJECT_ROOT / 'outputs')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [20]:

val_dataset = LidarDataset(MANIFEST_PATH, split='val', representation=REPRESENTATION, augment=False)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

model = MutliRepUnet(out_channels=3).to(DEVICE)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)

model.load_state_dict(checkpoint['model_state_dict'])

model.eval()

print('Loaded checkpoint:', CHECKPOINT_PATH)
print('Checkpoint epoch:', checkpoint['epoch'])


Loaded checkpoint: /home/jacobrawlins/Lidar_Project/checkpoints/multi_rep_unet.pt
Checkpoint epoch: 29


In [21]:

@torch.no_grad()
def collect_attention(model, loader):

    model.eval()

    all_attention = []
    aguada_presence = []

    for images, masks in tqdm(loader, desc='Collecting attention', unit='batch'):

        images = images.to(DEVICE, dtype=torch.float32)

        logits, attention_weights = model(images, return_attention=True)

        batch_weights = torch.stack(
            [
                weights.reshape(images.size(0), 3)
                for weights in attention_weights
            ],
            dim=1
        )

        all_attention.append(batch_weights.cpu())

        has_aguada = ((masks[:, 2] > 0.5).flatten(start_dim=1).any(dim=1))

        aguada_presence.append(has_aguada.cpu())
    attention = torch.cat(all_attention,dim=0).numpy()

    aguada_presence = torch.cat(aguada_presence, dim=0).numpy()

    return attention, aguada_presence
def save_figure(filename):
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=220, bbox_inches='tight')

    plt.close()

## Attention Heatmap

In [22]:

def plot_attention_heatmap(attention):
    mean_attention = attention.mean(axis=0)
    fig, ax = plt.subplots(figsize=(8, 5))

    heatmap = ax.imshow(mean_attention, cmap='viridis', vmin=0, vmax=1,aspect='auto')
    ax.set_xticks(range(3), REPRESENTATION_NAMES)
    ax.set_yticks(range(4), [f'Level {i + 1}' for i in range(4)])

    for level in range(4):
        for channel in range(3):
            value = mean_attention[level, channel]
            ax.text(channel, level, f'{value:.3f}', ha='center', va='center', color='white' if value < 0.45 else 'black')
    ax.set_title('Average Representation Attention by Encoder Level')

    fig.colorbar(heatmap, ax=ax, label='Attention Weight')

    save_figure('attention_heatmap.png')
    


In [23]:

def plot_stacked_attention(attention):

    mean_attention = attention.mean(axis=0)

    levels = ['Level 1', 'Level 2', 'Level 3', 'Level 4']

    bottom = np.zeros(4)
    fig, ax = plt.subplots(figsize=(9, 5))

    for channel, name in enumerate(REPRESENTATION_NAMES):
        values = mean_attention[:, channel]

        ax.bar(levels, values, bottom=bottom, label=name)

        bottom += values

    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Attention Weight')
    ax.set_xlabel('Encoder Level')

    ax.set_title('Distribution of Representation Attention')

    ax.legend()

    save_figure('stacked_attention.png')


In [24]:

def plot_attention_distributions(attention):

    fig, ax = plt.subplots(figsize=(12, 6))

    values = []
    positions = []
    for level in range(4):
        for channel in range(3):
            values.append(attention[:, level, channel])

            positions.append(level * 4 + channel + 1)

    boxplots = ax.boxplot(values, positions=positions, widths=0.65, patch_artist=True,showfliers=False)

    colors = ['steelblue', 'orange', 'green']

    for i, box in enumerate(boxplots['boxes']):
        box.set_facecolor(colors[i % 3])
        box.set_alpha(0.7)

    ax.set_xticks([2, 6, 10, 14], ['Level 1', 'Level 2', 'Level 3', 'Level 4'])

    ax.set_ylim(0, 1)
    ax.set_ylabel('Attention Weight')
    ax.set_title('Attention Weight Distribution Across Validation Tiles')

    ax.legend(handles=[Patch(color=colors[i], label=name) for i, name in enumerate(REPRESENTATION_NAMES)])

    save_figure('attention_distributions.png')


In [25]:

def plot_per_tile_attention(attention):

    level = 3

    openness_weights = attention[:, level, 1]

    sorted_indices = np.argsort(
        openness_weights
    )[::-1]

    selected_indices = sorted_indices[:60]

    fig, ax = plt.subplots(
        figsize=(11, 5)
    )

    for channel, name in enumerate(REPRESENTATION_NAMES):

        values = attention[selected_indices, level, channel]

        ax.plot(range(len(selected_indices)), values, label=name)

    ax.set_xlabel('Validation Tiles Sorted by Openness Attention')

    ax.set_ylabel('Attention Weight')

    ax.set_title('Representation Attention Across Selected Tiles - Level 4')

    ax.set_ylim(0, 1)
    ax.legend()

    save_figure('per_tile_attention.png')


In [26]:

def plot_aguada_attention(attention, aguada_presence):

    present = attention[aguada_presence]
    absent = attention[~aguada_presence]

    if len(present) == 0 or len(absent) == 0:
        print('Insufficient tiles for Aguada comparison.')
        return

    present_mean = present.mean(axis=0)
    absent_mean = absent.mean(axis=0)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

    levels = np.arange(4)

    for channel, name in enumerate(REPRESENTATION_NAMES):

        axes[channel].plot(levels, present_mean[:, channel], marker='o', label='Aguada Present')

        axes[channel].plot(levels, absent_mean[:, channel], marker='s', label='Aguada Absent')

        axes[channel].set_title(name)
        axes[channel].set_xticks(levels, ['L1', 'L2', 'L3', 'L4'])

        axes[channel].set_ylim(0, 1)
        axes[channel].set_xlabel('Encoder Level')

    axes[0].set_ylabel('Average Attention Weight')
    axes[0].legend()

    fig.suptitle('Representation Attention With and Without Aguadas')

    save_figure('aguada_attention.png')

    print(f'Aguada-present tiles: {len(present)}')
    print(f'Aguada-absent tiles: {len(absent)}')


In [27]:

@torch.no_grad()
def run_ablation(model, loader):

    model.eval()

    scenarios = [
        ('All Representations', None),
        ('Without SVF', 0),
        ('Without Openness', 1),
        ('Without Slope', 2)
    ]

    results_rows = []

    for scenario_name, removed_channel in scenarios:

        total_tp = torch.zeros(3, device=DEVICE)
        total_fp = torch.zeros(3, device=DEVICE)
        total_fn = torch.zeros(3, device=DEVICE)

        for images, masks in tqdm(loader, desc=scenario_name, unit='batch'):

            images = images.to(DEVICE, dtype=torch.float32)

            masks = masks.to(DEVICE, dtype=torch.float32)

            if removed_channel is not None:
                images = images.clone()
                images[:, removed_channel] = 0

            logits = model(images)

            counts = segmentation_metrics(logits, masks, threshold=BEST_THRESHOLDS)

            total_tp += counts['tp']
            total_fp += counts['fp']
            total_fn += counts['fn']

        metrics = calculate_metrics(total_tp, total_fp, total_fn)

        row = {
            'scenario': scenario_name,
            'mean_dice': float(metrics['mean_dice']),
            'mean_iou': float(metrics['mean_iou'])
        }

        for c, name in enumerate(CLASS_NAMES):
            row[f'{name.lower()}_dice'] = (
                metrics['dice'][c].item()
            )

        results_rows.append(row)

    return results_rows


In [28]:

def plot_ablation(results_rows):

    scenario_names = [row['scenario'] for row in results_rows]

    x = np.arange(len(scenario_names))
    width = 0.23

    fig, ax = plt.subplots(figsize=(11, 6))

    for c, name in enumerate(CLASS_NAMES):

        dice_values = [row[f'{name.lower()}_dice'] for row in results_rows]

        ax.bar(x + (c - 1) * width, dice_values, width, label=name)

    ax.set_xticks(x, scenario_names, rotation=15)

    ax.set_ylabel('Dice Score')
    ax.set_ylim(0, 1)

    ax.set_title('Segmentation Performance Under Representation Ablation')

    ax.legend()

    save_figure('ablation_dice.png')


In [29]:

def save_results(attention, results_rows):

    mean_attention = attention.mean(axis=0)
    std_attention = attention.std(axis=0)

    attention_path = OUTPUT_DIR / 'attention_summary.csv'

    with open(attention_path, 'w', newline='') as file:

        writer = csv.writer(file)

        writer.writerow([
            'Encoder Level',
            'Representation',
            'Mean Attention',
            'Standard Deviation'
        ])

        for level in range(4):
            for channel, name in enumerate(REPRESENTATION_NAMES):
                writer.writerow([
                    level + 1,
                    name,
                    mean_attention[level, channel],
                    std_attention[level, channel]
                ])

    ablation_path = OUTPUT_DIR / 'ablation_results.csv'

    with open(ablation_path, 'w', newline='') as file:

        writer = csv.DictWriter(file, fieldnames=results_rows[0].keys())

        writer.writeheader()
        writer.writerows(results_rows)

    np.save(OUTPUT_DIR / 'attention_per_tile.npy', attention)

    print('Saved attention summary:', attention_path)
    print('Saved ablation results:', ablation_path)


In [30]:

def main():

    print('Collecting attention weights...')

    attention, aguada_presence = collect_attention(
        model,
        val_loader
    )

    print('Attention shape:', attention.shape)

    print('\nGenerating attention visualizations...')

    plot_attention_heatmap(attention)

    plot_stacked_attention(attention)

    plot_attention_distributions(attention)

    plot_per_tile_attention(attention)

    plot_aguada_attention(attention, aguada_presence)

    print('\nRunning representation ablation...')

    ablation_results = run_ablation(model, val_loader)

    plot_ablation(ablation_results)

    save_results(attention, ablation_results)

    print('\nAnalysis complete.')
    print('Results saved to:', OUTPUT_DIR)


if __name__ == '__main__':
    main()


/home/jacobrawlins/Lidar_Project/.venv/lib/python3.12/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


Attention shape: (314, 4, 3)

Generating attention visualizations...
Aguada-present tiles: 11
Aguada-absent tiles: 303

Running representation ablation...


All Representations:   0%|          | 0/157 [00:00<?, ?batch/s]

/home/jacobrawlins/Lidar_Project/.venv/lib/python3.12/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


Without SVF:   0%|          | 0/157 [00:00<?, ?batch/s]

Without Openness:   0%|          | 0/157 [00:00<?, ?batch/s]

Without Slope:   0%|          | 0/157 [00:00<?, ?batch/s]

Saved attention summary: /home/jacobrawlins/Lidar_Project/outputs/attention_summary.csv
Saved ablation results: /home/jacobrawlins/Lidar_Project/outputs/ablation_results.csv

Analysis complete.
Results saved to: /home/jacobrawlins/Lidar_Project/outputs
